# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [4]:
import json
import requests
import pandas as pd

# Using Guardian API to search for articles related to the commodities
with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']

BASE_URL = 'https://content.guardianapis.com'

all_results = []

for page in range(1, 20):
    parameters = {
        "api-key": API_KEY,
        "q": "oil OR natural gas OR copper OR wheat OR OPEC OR energy crisis",
        "page-size": 200,
        "page": page,
        "show-fields": "bodyText"
    }
    response = requests.get(f"{BASE_URL}/search", params=parameters)
    response.raise_for_status()
    data = response.json()
    results = data['response']['results']
    if not results:
        break

    for article in results:
        all_results.append({
            'date': article.get('webPublicationDate'),
            'section': article.get('sectionName'),
            'title': article.get('webTitle'),
            'body': article.get('fields',{}).get('bodyText', '')
        })

df = pd.DataFrame(all_results)
df.to_csv('data/guardian_commodities.csv')
print(df.head(10))



                   date      section  \
0  2026-04-05T18:22:38Z     Business   
1  2026-03-24T15:54:39Z  Environment   
2  2026-04-07T16:42:27Z     Business   
3  2026-03-18T18:51:03Z   World news   
4  2026-03-24T04:36:06Z   World news   
5  2026-03-13T01:29:13Z   World news   
6  2026-03-26T16:49:13Z     Business   
7  2026-02-05T13:00:37Z      US news   
8  2026-03-23T15:00:53Z  Environment   
9  2026-03-01T14:21:14Z     Business   

                                               title  \
0  Iran strikes Kuwait’s oil infrastructure befor...   
1  Green energy boss backs more North Sea oil and...   
2  Oil and gas crisis from Iran war worse than 19...   
3  Damaged Russian tanker carrying natural gas fl...   
4  Japan to begin biggest-ever oil release from n...   
5  Ukraine war briefing: Putin envoy says US bett...   
6  There are solutions to Britain’s energy crisis...   
7  Michigan accuses big oil of being ‘cartel’ tha...   
8  Europe’s ‘staggering’ clean power gains underm...   

In [8]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
commodities = {
    "Gold":        "GC=F",   # Gold Futures
    "Silver":      "SI=F",   # Silver Futures
    "Crude Oil":   "CL=F",   # WTI Crude Oil Futures
    "Brent Oil":   "BZ=F",   # Brent Crude Futures
    "Natural Gas": "NG=F",   # Natural Gas Futures
    "Copper":      "HG=F",   # Copper Futures
    "Wheat":       "ZW=F",   # Wheat Futures
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2014-01-01",
    end="2026-03-31",
    interval="1d"       # 1d; daily
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head())


[*********************100%***********************]  7 of 7 completed

                  Gold     Silver    Crude Oil  Brent Oil  Natural Gas  \
Date                                                                     
2014-01-02  107.779999  95.440002  1225.000000     3.4315        4.321   
2014-01-03  106.889999  93.959999  1238.400024     3.4060        4.304   
2014-01-06  106.730003  93.430000  1237.800049     3.4120        4.306   
2014-01-07  107.349998  93.669998  1229.400024     3.4110        4.299   
2014-01-08  107.150002  92.330002  1225.300049     3.3935        4.216   

               Copper   Wheat  
Date                           
2014-01-02  20.098000  597.00  
2014-01-03  20.181999  605.75  
2014-01-06  20.077000  605.75  
2014-01-07  19.764999  602.50  
2014-01-08  19.518000  588.75  
